# U4-02 · Redes convolucionales con las células de la sangre

*Especialista en Inteligencia Artificial (IFCD107) · Unidad 4*

Ejecuta cada celda con **Mayús + Intro** (en Colab, también con ▶). Con GPU (*Entorno de ejecución → Cambiar tipo de entorno de ejecución*) las redes se entrenan mucho más deprisa.

> Datos: BloodMNIST (MedMNIST v2, CC BY 4.0). Yang, J. et al. (2023). *Scientific Data*, 10, 41. https://doi.org/10.1038/s41597-022-01721-8, a partir de Acevedo, A. et al. (2020). *Data in Brief*, 30, 105474. https://doi.org/10.1016/j.dib.2020.105474. No apto para uso clínico.

In [ ]:
# Descarga los datos y módulos auxiliares (solo la primera vez)
import os, urllib.request
for f in ["bloodmnist.npz", "sangre.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve("https://machinelearning-formacion.github.io/ifcd107/recursos/datos/u4/" + f, f)

## 4.3 · Redes convolucionales: células de la sangre

`n1_sangre.py`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sangre import X_e, y_e, X_v, X_p, y_p, nombres

print("entrenamiento", X_e.shape, "| validación", X_v.shape,
      "| prueba", X_p.shape)
n = np.bincount(y_e)
for c, k in enumerate(n):
    print(f"{c} {nombres[c]:<21}{k:5d}  ({k / n.sum():.1%})")
print("media de cada canal (R, G, B):", X_e.mean((0, 1, 2)).round(3))
corto = ["basófilo", "eosinófilo", "eritrobl.", "gran. inm.",
         "linfocito", "monocito", "neutrófilo", "plaqueta"]
fig, ejes = plt.subplots(2, 8, figsize=(4.6, 1.3))
for c in range(8):
    for f, i in enumerate(np.flatnonzero(y_e == c)[:2]):
        ejes[f, c].imshow(X_e[i]); ejes[f, c].axis("off")
    ejes[0, c].set_title(corto[c], fontsize=5.5)
plt.show()

`n3_convolucion.py`

In [ ]:
import numpy as np
from sangre import X_e

img = X_e[0].mean(2)                      # gris: media de R, G y B
K = np.array([[-1, 0, 1],      # filtro de bordes verticales
              [-2, 0, 2],
              [-1, 0, 1]])
def convolucion(img, K):
    h, w = img.shape[0] - 2, img.shape[1] - 2
    S = np.zeros((h, w))
    for i in range(h):
        for j in range(w):                 # suma de productos 3x3
            S[i, j] = (img[i:i + 3, j:j + 3] * K).sum()
    return S
S = convolucion(img, K)
print("imagen", img.shape, "-> mapa", S.shape)
print("fila 14 de la imagen:", img[14, 4:14].round(2))
print("fila 13 del mapa:    ", S[13, 3:13].round(2))
print("parámetros de una capa Conv2D(32, 3) sobre RGB:",
      3 * 3 * 3 * 32 + 32)
print("parámetros de una capa Dense(32) sobre 28x28x3:",
      28 * 28 * 3 * 32 + 32)

`n2_mlp_cnn.py`

In [ ]:
import time
import keras
from keras import layers
from sangre import X_e, y_e, X_v, y_v, X_p, y_p

def probar(nombre, capas):
    keras.utils.set_random_seed(0)
    m = keras.Sequential([keras.Input((28, 28, 3))] + capas
                         + [layers.Dense(8, activation="softmax")])
    m.compile("adam", "sparse_categorical_crossentropy",
              metrics=["accuracy"])
    t = time.perf_counter()
    m.fit(X_e, y_e, epochs=10, batch_size=128, verbose=0,
          validation_data=(X_v, y_v))
    s = time.perf_counter() - t
    print(f"{nombre:<14} parámetros {m.count_params():7d}"
          f"  acierto prueba {m.evaluate(X_p, y_p, verbose=0)[1]:.3f}"
          f"  ({s:.0f} s)")
probar("densa", [layers.Flatten(),
                 layers.Dense(128, activation="relu"),
                 layers.Dense(64, activation="relu")])
probar("convolucional", [
    layers.Conv2D(32, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Flatten(), layers.Dense(64, activation="relu")])

`n4_mejor_cnn.py`

In [ ]:
import keras
from keras import layers
from sangre import X_e, y_e, X_v, y_v, X_p, y_p

keras.utils.set_random_seed(0)
def bloque(filtros):
    return [layers.Conv2D(filtros, 3, padding="same",
                          use_bias=False),
            layers.BatchNormalization(), layers.Activation("relu"),
            layers.Conv2D(filtros, 3, padding="same",
                          activation="relu"),
            layers.MaxPooling2D()]
m = keras.Sequential(
    [keras.Input((28, 28, 3)),
     layers.RandomFlip("horizontal_and_vertical"),   # aumento
     layers.RandomRotation(0.25)]
    + bloque(32) + bloque(64) + bloque(128)
    + [layers.GlobalAveragePooling2D(), layers.Dropout(0.3),
       layers.Dense(8, activation="softmax")])
m.compile(keras.optimizers.Adam(0.001),
          "sparse_categorical_crossentropy", metrics=["accuracy"])
cb = [keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3),
      keras.callbacks.EarlyStopping(patience=8,
                                    restore_best_weights=True)]
h = m.fit(X_e, y_e, epochs=30, batch_size=128, verbose=0,
          validation_data=(X_v, y_v), callbacks=cb)
a = h.history
for e in [0, 4, 9, 19, len(a["loss"]) - 1]:
    print(f"época {e + 1:2d}: acierto entr. {a['accuracy'][e]:.3f}"
          f"  val. {a['val_accuracy'][e]:.3f}")
print("parámetros:", m.count_params())
print(f"acierto prueba: {m.evaluate(X_p, y_p, verbose=0)[1]:.3f}")
m.save("cnn_sangre.keras")

`n5_errores.py`

In [ ]:
import numpy as np
import pandas as pd
import keras
from sklearn.metrics import confusion_matrix, f1_score
from sangre import X_p, y_p, nombres

m = keras.models.load_model("cnn_sangre.keras")
P = m.predict(X_p, verbose=0)
pred = P.argmax(1)
ab = [n[:5] for n in nombres]
print(pd.DataFrame(confusion_matrix(y_p, pred), ab, ab).to_string())
f1 = f1_score(y_p, pred, average=None)
print("F1 por clase:", dict(zip(ab, f1.round(3).tolist())))
seguro = P.max(1)
for u in [0.5, 0.9, 0.99]:
    s = seguro >= u
    print(f"confianza >= {u}: {s.mean():.1%} de las imágenes,"
          f" acierto {(pred[s] == y_p[s]).mean():.3f}")

`n7_mapas.py`

In [ ]:
import keras
import matplotlib.pyplot as plt
from sangre import X_p

m = keras.models.load_model("cnn_sangre.keras")
conv = [c for c in m.layers if "conv2d" in c.name][0]
W = conv.get_weights()[0]                 # (3, 3, 3, 32)
print("filtros de la primera capa:", W.shape)
mapas = keras.Model(m.inputs, conv.output).predict(X_p[:1],
                                                   verbose=0)[0]
print("mapas de activación:", mapas.shape)
fig, ejes = plt.subplots(2, 8, figsize=(4.6, 1.2))
ejes[0, 0].imshow(X_p[0])
for k in range(15):
    e = ejes.flat[k + 1]
    e.imshow(mapas[:, :, k], cmap="gray")
for e in ejes.flat:
    e.axis("off")
plt.show()

`n9_oclusion.py`

In [ ]:
import numpy as np
import keras
import matplotlib.pyplot as plt
from sangre import X_p, y_p, nombres

m = keras.models.load_model("cnn_sangre.keras")
P = m.predict(X_p, verbose=0)
fig, ejes = plt.subplots(2, 4, figsize=(4.6, 2.4))
for col, c in enumerate([1, 2, 6, 7]):    # una célula de 4 clases
    i = np.flatnonzero((y_p == c) & (P[:, c] > 0.9))[0]  # acertada
    img = X_p[i]
    p0, gris = P[i, c], img.mean((0, 1))
    tapadas = []
    for f in range(0, 28, 4):              # tapar bloques de 8x8
        for k in range(0, 28, 4):
            t = img.copy()
            t[max(f - 2, 0):f + 6, max(k - 2, 0):k + 6] = gris
            tapadas.append(t)
    p = m.predict(np.array(tapadas), verbose=0)[:, c].reshape(7, 7)
    print(f"{nombres[c]:<11} prob. {p0:.3f}; al tapar, mínima"
          f" {p.min():.3f}")
    ejes[0, col].imshow(img); ejes[1, col].imshow(p0 - p, cmap="Reds")
    ejes[0, col].set_title(nombres[c], fontsize=6)
for e in ejes.flat:
    e.axis("off")
plt.show()

`n6_regularizar.py`

In [ ]:
import numpy as np
import keras
from keras import layers
from sangre import X_e, y_e, X_p, y_p

i = np.random.default_rng(0).choice(len(X_e), 1000, replace=False)
pocas, y_pocas = X_e[i], y_e[i]           # solo 1000 imágenes
def cnn(extra):
    keras.utils.set_random_seed(0)
    m = keras.Sequential(
        [keras.Input((28, 28, 3))] + extra
        + [layers.Conv2D(32, 3, activation="relu", padding="same"),
           layers.MaxPooling2D(),
           layers.Conv2D(64, 3, activation="relu", padding="same"),
           layers.MaxPooling2D(), layers.Flatten(),
           layers.Dropout(0.5 if extra else 0.0),
           layers.Dense(8, activation="softmax")])
    m.compile("adam", "sparse_categorical_crossentropy",
              metrics=["accuracy"])
    m.fit(pocas, y_pocas, epochs=60, batch_size=64, verbose=0)
    return (m.evaluate(pocas, y_pocas, verbose=0)[1],
            m.evaluate(X_p, y_p, verbose=0)[1])
for nombre, extra in [
        ("sin regularizar", []),
        ("aumento + dropout",
         [layers.RandomFlip("horizontal_and_vertical"),
          layers.RandomRotation(0.25)])]:
    e, p = cnn(extra)
    print(f"{nombre:<18} acierto entr. {e:.3f}  prueba {p:.3f}")

`n8_transferencia.py`

In [ ]:
import numpy as np
import keras
from keras import layers
from sklearn.metrics import balanced_accuracy_score
from sangre import X_e, y_e, X_p, y_p, nombres

def base_nueva():
    return keras.Sequential([
        keras.Input((28, 28, 3)),
        layers.Conv2D(32, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(), layers.GlobalAveragePooling2D()])
def ajustar(base, entrenable, Xn, yn, Xq, yq):
    base.trainable = entrenable
    m = keras.Sequential([base,
                          layers.Dense(1, activation="sigmoid")])
    m.compile(keras.optimizers.Adam(1e-3 if entrenable else 1e-2),
              "binary_crossentropy")
    m.fit(Xn, yn, epochs=40, batch_size=20, verbose=0)
    p = m.predict(Xq, verbose=0).ravel() > 0.5
    return balanced_accuracy_score(yq, p)
for nuevas in [[4, 5], [1, 6]]:       # clases que la base no verá
    vieja = ~np.isin(y_e, nuevas)
    mapa = {c: k for k, c in enumerate(sorted(set(y_e[vieja])))}
    keras.utils.set_random_seed(0)
    base = base_nueva()                   # preentrenar con 6 clases
    previa = keras.Sequential([base, layers.Dense(6, "softmax")])
    previa.compile("adam", "sparse_categorical_crossentropy")
    previa.fit(X_e[vieja], np.vectorize(mapa.get)(y_e[vieja]),
               epochs=10, batch_size=128, verbose=0)
    rng = np.random.default_rng(0)        # solo 50 imágenes por clase
    i = np.concatenate([rng.choice(np.flatnonzero(y_e == c), 50,
                                   replace=False) for c in nuevas])
    Xn, yn = X_e[i], (y_e[i] == nuevas[1]).astype(int)
    j = np.isin(y_p, nuevas)
    Xq, yq = X_p[j], (y_p[j] == nuevas[1]).astype(int)
    print(f"{nombres[nuevas[0]]} frente a {nombres[nuevas[1]]}"
          f" (acierto equilibrado):")
    keras.utils.set_random_seed(1)
    print(f"  desde cero      "
          f" {ajustar(base_nueva(), True, Xn, yn, Xq, yq):.3f}")
    print(f"  base congelada  "
          f" {ajustar(base, False, Xn, yn, Xq, yq):.3f}")

## 4.7 · Ajustar y poner en producción

`r2_guardar.py`

In [ ]:
import os
import time
import numpy as np
import keras
from sangre import X_p, nombres

m = keras.models.load_model("cnn_sangre.keras")
mb = os.path.getsize("cnn_sangre.keras") / 1e6
print(f"tamaño del archivo: {mb:.1f} MB")
def clasificar(imagen):                   # lo que haría un servicio
    p = m.predict(imagen[None], verbose=0)[0]
    return {"clase": nombres[p.argmax()], "probabilidad":
            round(float(p.max()), 3)}
print(clasificar(X_p[0]))
t = time.perf_counter()
for i in range(50):
    clasificar(X_p[i])
una = (time.perf_counter() - t) / 50
t = time.perf_counter(); m.predict(X_p[:1000], verbose=0)
lote = (time.perf_counter() - t) / 1000
print(f"una a una: {una * 1000:.1f} ms por imagen;"
      f" en lote: {lote * 1000:.2f} ms por imagen")

**Prueba.** En `n4_mejor_cnn.py`, quita el aumento de datos. ¿Cuánto cambia el acierto de prueba?

**Ejercicio.** En `n8_transferencia.py`, prueba con otro par de clases que la base no haya visto.